# 🔍 DefectVision — 03. Entraînement YOLOv8

**Objectif** : Entraîner YOLOv8 pour détecter les défauts industriels sur MVTec AD (bottle).

**Workflow** :
```
dataset.yaml (Drive) → YOLOv8n → Entraînement T4 → Évaluation → HuggingFace Hub
```

## 0. Setup

In [ ]:
# Vérifier le GPU
!nvidia-smi

In [ ]:
!pip install -q ultralytics huggingface_hub

In [ ]:
# Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

DRIVE_BASE  = Path('/content/drive/MyDrive/defect-vision')
YOLO_PATH   = DRIVE_BASE / 'data' / 'yolo' / 'bottle'
YAML_PATH   = YOLO_PATH / 'dataset.yaml'
MODELS_PATH = DRIVE_BASE / 'models'
MODELS_PATH.mkdir(parents=True, exist_ok=True)

print(f'✅ dataset.yaml : {YAML_PATH}')
print(f'✅ Models path  : {MODELS_PATH}')
assert YAML_PATH.exists(), '❌ dataset.yaml introuvable — lance le notebook 02 dabord'

In [ ]:
# Vérifier le contenu du dataset.yaml
with open(YAML_PATH) as f:
    print(f.read())

## 1. Configuration de l'entraînement

In [ ]:
# Hyperparamètres
CONFIG = {
    'model'     : 'yolov8n.pt',   # nano — rapide sur T4
    'epochs'    : 50,
    'imgsz'     : 640,
    'batch'     : 16,
    'patience'  : 10,             # early stopping
    'lr0'       : 0.01,
    'project'   : '/content/runs',
    'name'      : 'yolov8_bottle',
    'device'    : 0,              # GPU
    'workers'   : 2,
    'exist_ok'  : True,
    'pretrained': True,
    'augment'   : True,
    'cache'     : False,
}

print('✅ Configuration :')
for k, v in CONFIG.items():
    print(f'  {k:12s} : {v}')

## 2. Entraînement

In [ ]:
from ultralytics import YOLO
import time

# Charger le modèle YOLOv8 nano pré-entraîné
model = YOLO(CONFIG['model'])
print(f'✅ Modèle chargé : {CONFIG["model"]}')

In [ ]:
# Lancer l'entraînement
print('🚀 Démarrage de l entraînement...')
start = time.time()

results = model.train(
    data     = str(YAML_PATH),
    epochs   = CONFIG['epochs'],
    imgsz    = CONFIG['imgsz'],
    batch    = CONFIG['batch'],
    patience = CONFIG['patience'],
    lr0      = CONFIG['lr0'],
    project  = CONFIG['project'],
    name     = CONFIG['name'],
    device   = CONFIG['device'],
    workers  = CONFIG['workers'],
    exist_ok = CONFIG['exist_ok'],
    pretrained = CONFIG['pretrained'],
    augment  = CONFIG['augment'],
    cache    = CONFIG['cache'],
)

elapsed = time.time() - start
print(f'\n✅ Entraînement terminé en {elapsed/60:.1f} minutes')

## 3. Résultats et métriques

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

RUN_PATH = Path(CONFIG['project']) / CONFIG['name']

# Afficher les courbes d'entraînement
results_png = RUN_PATH / 'results.png'
if results_png.exists():
    img = mpimg.imread(str(results_png))
    plt.figure(figsize=(18, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Courbes d entraînement YOLOv8', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Courbes sauvegardées')

In [ ]:
# Afficher la matrice de confusion
confusion_png = RUN_PATH / 'confusion_matrix.png'
if confusion_png.exists():
    img = mpimg.imread(str(confusion_png))
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Matrice de confusion', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Évaluation sur le test set
print('📊 Évaluation sur le test set...')
metrics = model.val(
    data   = str(YAML_PATH),
    split  = 'test',
    device = 0,
)

print('\n=== Métriques Test Set ===')
print(f'  mAP50      : {metrics.box.map50:.4f}')
print(f'  mAP50-95   : {metrics.box.map:.4f}')
print(f'  Precision  : {metrics.box.mp:.4f}')
print(f'  Recall     : {metrics.box.mr:.4f}')

## 4. Inférence — visualisation des prédictions

In [ ]:
import cv2
import numpy as np
import random

# Charger le meilleur modèle
best_model = YOLO(str(RUN_PATH / 'weights' / 'best.pt'))

# Prendre 6 images du test set
test_imgs = list((YOLO_PATH / 'test' / 'images').glob('*.png'))
samples   = random.sample(test_imgs, min(6, len(test_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('YOLOv8 — Prédictions sur le test set', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, img_path in enumerate(samples):
    # Inférence
    preds = best_model.predict(str(img_path), conf=0.25, verbose=False)
    result_img = preds[0].plot()  # image avec bboxes dessinées
    result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)

    n_det = len(preds[0].boxes)
    axes[i].imshow(result_img)
    axes[i].set_title(f'{img_path.stem} — {n_det} détection(s)', fontsize=9)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('yolov8_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Prédictions visualisées')

## 5. Sauvegarde sur Drive

In [ ]:
import shutil

# Sauvegarder best.pt et last.pt sur Drive
dest = MODELS_PATH / 'yolov8_bottle'
dest.mkdir(parents=True, exist_ok=True)

shutil.copy2(RUN_PATH / 'weights' / 'best.pt', dest / 'best.pt')
shutil.copy2(RUN_PATH / 'weights' / 'last.pt', dest / 'last.pt')

# Sauvegarder aussi les courbes
for png in ['results.png', 'confusion_matrix.png']:
    src = RUN_PATH / png
    if src.exists():
        shutil.copy2(src, dest / png)

print(f'✅ Modèle sauvegardé sur Drive : {dest}')
print(f'  best.pt : {(dest / "best.pt").stat().st_size / 1e6:.1f} MB')

## 6. Push sur HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi, login

# ⚠️ Remplace par ton token HuggingFace
# Génère-le sur : https://huggingface.co/settings/tokens
HF_TOKEN    = 'TON_TOKEN_HUGGINGFACE'
HF_USERNAME = 'Chasston'
REPO_NAME   = 'defect-vision-yolov8-bottle'

login(token=HF_TOKEN)
api = HfApi()

# Créer le repo si besoin
repo_id = f'{HF_USERNAME}/{REPO_NAME}'
api.create_repo(repo_id=repo_id, repo_type='model', exist_ok=True)
print(f'✅ Repo HuggingFace : {repo_id}')

In [ ]:
# Créer un README pour HuggingFace
readme_content = f"""---
tags:
- object-detection
- computer-vision
- yolov8
- defect-detection
- industrial
datasets:
- mvtec-ad
---

# DefectVision — YOLOv8 Defect Detection (bottle)

YOLOv8n fine-tuned on MVTec AD dataset for industrial defect detection.

## Category : bottle
- **Defect types** : broken_large, broken_small, contamination
- **Classes** : 1 (defect)
- **Image size** : 640x640

## Metrics (test set)
- mAP50     : {metrics.box.map50:.4f}
- mAP50-95  : {metrics.box.map:.4f}
- Precision : {metrics.box.mp:.4f}
- Recall    : {metrics.box.mr:.4f}

## Usage
```python
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

path  = hf_hub_download(repo_id='{repo_id}', filename='best.pt')
model = YOLO(path)
results = model.predict('your_image.png', conf=0.25)
```

## Author
[{HF_USERNAME}](https://huggingface.co/{HF_USERNAME})
"""

with open('/content/README.md', 'w') as f:
    f.write(readme_content)

print('✅ README généré')

In [ ]:
# Uploader les fichiers sur HuggingFace
files_to_upload = [
    (dest / 'best.pt',             'best.pt'),
    (dest / 'results.png',         'results.png'),
    ('/content/README.md',         'README.md'),
]

for local_path, hf_filename in files_to_upload:
    if Path(local_path).exists():
        api.upload_file(
            path_or_fileobj = str(local_path),
            path_in_repo    = hf_filename,
            repo_id         = repo_id,
            repo_type       = 'model',
        )
        print(f'  ✅ Uploadé : {hf_filename}')

print(f'\n🎉 Modèle disponible sur : https://huggingface.co/{repo_id}')

## ✅ Résumé

| Info | Valeur |
|------|--------|
| Modèle | YOLOv8n fine-tuned |
| Dataset | MVTec AD — bottle |
| Drive | `/defect-vision/models/yolov8_bottle/` |
| HuggingFace | `Chasston/defect-vision-yolov8-bottle` |
| Prochain notebook | `04_patchcore.ipynb` |

**Prochaine étape** : Entraîner PatchCore (détection d'anomalies non supervisée).